# Data Preparation

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 23.04.2024
- Date of Last Modification: 24.04.2024

## 1. Setup

### 1.1 Import Libraries

In [ ]:
import os
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import pyensembl
import scanpy as sc
import squidpy as sq

import nichejepa

### 1.2 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

Run ```pyensembl install --release 111 --species homo_sapiens```.

In [ ]:
ensembl = pyensembl.EnsemblRelease(111)

### 1.3 Configure Paths and Create Directories

In [ ]:
# Define paths
st_data_folder_path = "../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"

# Create required directories
os.makedirs(st_data_bronze_folder_path, exist_ok=True)
os.makedirs(st_data_silver_folder_path, exist_ok=True)
os.makedirs(st_data_gold_folder_path, exist_ok=True)

## 2. Manuscript Data

### 2.1 Nanostring CosMx Human Brain

- Data Access: https://nanostring.com/products/cosmx-spatial-molecular-imager/ffpe-dataset/human-frontal-cortex-ffpe-dataset/

In [ ]:
# Read data
df = pd.read_csv(f"{st_data_bronze_folder_path}/S3_exprMat_file.csv.gz")
df_metadata = pd.read_csv(f"{st_data_bronze_folder_path}/S3_metadata_file.csv.gz")

In [ ]:
## Create adata object
# Only keep genes
genes = [col for col in df.columns if
         ("FalseCode" not in col) &
         ("NegPrb" not in col) &
         ("Custom" not in col) &
         ("cell" not in col) &
         ("fov" not in col)]
df_counts = df[genes]

genes = [gene.split("/")[0] for gene in df_counts.columns]

# Rename some genes with names from Ensembl database
for old_gene_name, new_gene_name in zip(["COX1", "COX2", "CT45A", "HLA-DRB", "LOC728392", "METTL7A", "SLC9A3R1", "TTC30A"],
                                        ["MT-CO1", "PTGS2", "CT45A1", "HLA-DRB1", "ENSG00000286190", "TMT1A", "NHERF1", "IFT70A"]):
    idx = genes.index(old_gene_name)
    genes[idx] = new_gene_name
    
df_counts.columns = genes

var_metadata = pd.DataFrame(index=df_counts.columns)
obs_metadata = pd.DataFrame(index=df_counts.index, data={"fov": df["fov"].values,
                                                         "cell_id": df["cell_ID"].values,
                                                         "cell": df["cell"].values})

adata = ad.AnnData(X=df_counts.values, obs=obs_metadata, var=var_metadata)

adata.obs["niche_type"] = df_metadata["spatialClusteringAssignments"].values
adata.obs["cell_type"] = df_metadata["RefinedClusts_Final"].values

adata.obsm["spatial"] = np.array(df_metadata[["CenterX_global_px", "CenterY_global_px"]])

# Get ensembl ids
genes = adata.var.index.tolist()
ensembl_ids = []
for gene in genes:
    try:
        ensembl_ids.append(ensembl.gene_ids_of_gene_name(gene)[0])
    except ValueError as e:
        ensembl_ids.append(gene)
        print(f"Gene {gene} not found in Ensembl release.")
adata.var["ensembl_id"] = ensembl_ids

# Compute spatial neighborhood
n_neighbors = 8
spatial_key = "spatial"

sq.gr.spatial_neighbors(adata,
                        coord_type="generic",
                        spatial_key=spatial_key,
                        n_neighs=n_neighbors)

# Compute counts per niche
adata.layers["counts_niche"] = np.array(
    (adata.obsp["spatial_connectivities"].T @ adata.X) /
     adata.obsp["spatial_connectivities"].sum(0).T)

adata.layers["counts"] = adata.X

adata.write(f"{st_data_silver_folder_path}/nanostring_cosmx_human_brain.h5ad")

In [ ]:
adata = sc.read_h5ad("../datasets/st_data/gold/nanostring_cosmx_human_brain.h5ad")

In [ ]:
sc.pp.pca(adata)

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata,
           color=["cell_type"])

In [ ]:
sq.pl.spatial_scatter(adata, color="niche", shape=None) 

In [ ]:
sq.pl.spatial_scatter(adata, color="cell_type", shape=None) 